In [1]:
!pip install openai pydantic python-dotenv pandas


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import openai
import pydantic
import dotenv
import pandas as pd

print("OK")

OK


In [3]:
!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2134  100  2134    0     0   9892      0 --:--:-- --:--:-- --:--:--  9925
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3073  100  3073    0     0  14968      0 --:--:-- --:--:-- --:--:-- 14990


In [4]:
!ls

Untitled.ipynb         evaluation_utils.py    llm-zoomcamp-hw3.ipynb
__pycache__            llm-zoomcamp-hw1.ipynb models
download.py            llm-zoomcamp-hw2.ipynb rag_helper.py
embedder.py            llm-zoomcamp-hw3


In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [6]:
len(documents)

72

In [11]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

api_key is not None

False

In [12]:
import os

os.getcwd()

'/Users/kati/Desktop/agentic-rag-homework'

In [13]:
import os

os.listdir()

['.DS_Store',
 'llm-zoomcamp-hw1.ipynb',
 'llm-zoomcamp-hw3.ipynb',
 'evaluation_utils.py',
 'llm-zoomcamp-hw4.ipynb',
 'download.py',
 'llm-zoomcamp-hw2.ipynb',
 'models',
 '__pycache__',
 'rag_helper.py',
 '.gitignore',
 '.venv',
 'embedder.py',
 '.ipynb_checkpoints',
 '.git',
 'llm-zoomcamp-hw3']

In [14]:
!ls -la

total 176
drwxr-xr-x  19 kati  staff    608 Jun 28 12:26 .
drwx------@ 52 kati  staff   1664 Jun 28 12:26 ..
-rw-r--r--@  1 kati  staff   6148 Jun 27 20:07 .DS_Store
-rw-r--r--   1 kati  staff    182 Jun 28 12:26 .env
drwxr-xr-x  12 kati  staff    384 Jun 27 20:09 .git
-rw-r--r--   1 kati  staff     20 Jun 21 15:52 .gitignore
drwxr-xr-x@  7 kati  staff    224 Jun 28 12:18 .ipynb_checkpoints
drwxr-xr-x   9 kati  staff    288 Jun 21 16:05 .venv
drwxr-xr-x   3 kati  staff     96 Jun 27 16:28 __pycache__
-rw-r--r--   1 kati  staff   1376 Jun 27 16:08 download.py
-rw-r--r--   1 kati  staff   1520 Jun 27 16:08 embedder.py
-rw-r--r--   1 kati  staff   3073 Jun 28 12:16 evaluation_utils.py
-rw-r--r--@  1 kati  staff   8307 Jun 21 15:54 llm-zoomcamp-hw1.ipynb
-rw-r--r--   1 kati  staff  24630 Jun 27 17:44 llm-zoomcamp-hw2.ipynb
drwxr-xr-x   6 kati  staff    192 Jun 27 20:32 llm-zoomcamp-hw3
-rw-r--r--   1 kati  staff    617 Jun 27 17:51 llm-zoomcamp-hw3.ipynb
-rw-r--r--   1 kati  staff   9283 J

In [15]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("OPENAI_API_KEY") is not None)

True


In [20]:
import evaluation_utils

print(dir(evaluation_utils))

['RAGBase', 'RAGWithUsage', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calc_price', 'calc_total_price', 'llm_structured', 'llm_structured_retry', 'map_progress', 'time', 'tqdm']


In [21]:
from pydantic import BaseModel
from typing import List
from openai import OpenAI
from evaluation_utils import llm_structured

class Questions(BaseModel):
    questions: List[str]

client = OpenAI()

In [22]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [23]:
import inspect
from evaluation_utils import llm_structured

print(inspect.signature(llm_structured))

(client, instructions, user_prompt, output_type, model='gpt-5.4-mini')


### **Q1**

In [24]:
import json

first_3_docs = documents[:3]

generated_questions = []
token_usages = []

for doc in first_3_docs:
    user_prompt = json.dumps(
        {
            "filename": doc["filename"],
            "content": doc["content"],
        },
        ensure_ascii=False,
    )

    questions, usage = llm_structured(
        client=client,
        instructions=data_gen_instructions,
        user_prompt=user_prompt,
        output_type=Questions,
    )

    token_usages.append(usage)

    for question in questions.questions:
        generated_questions.append(
            {
                "question": question,
                "filename": doc["filename"],
            }
        )

    print(doc["filename"])
    print(questions.questions)
    print(usage)
    print()

01-agentic-rag/lessons/01-intro.md
['What exactly is a retrieval-augmented generation system, and why does it help with things like wrong or outdated answers?', 'Why does this course build the RAG project in plain Python instead of starting with a framework?', 'What are the main limits of large language models that this lesson points out?', 'What will the course project be, and what kind of question should the final agent be able to answer?', 'What are the two parts of this module, and what changes in the second part when the pipeline becomes more agent-like?']
ResponseUsage(input_tokens=1016, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=119, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1135)

01-agentic-rag/lessons/02-environment.md
['What do I need installed before starting this module besides Python itself?', 'How do I set up the project from scratch with uv and add the needed packages?', 'What’s the recommended way to store 

In [25]:
!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/04-evaluation/ground-truth.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 48627  100 48627    0     0   131k      0 --:--:-- --:--:-- --:--:--  131k


In [26]:
!ls

__pycache__            ground-truth.csv       llm-zoomcamp-hw3.ipynb
download.py            llm-zoomcamp-hw1.ipynb llm-zoomcamp-hw4.ipynb
embedder.py            llm-zoomcamp-hw2.ipynb models
evaluation_utils.py    llm-zoomcamp-hw3       rag_helper.py


In [27]:
import pandas as pd

df_ground_truth = pd.read_csv("ground-truth.csv")

df_ground_truth.head()

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md
2,What are the main weaknesses of large language...,01-agentic-rag/lessons/01-intro.md
3,What will the course build in the first part o...,01-agentic-rag/lessons/01-intro.md
4,What kind of example app are you building here...,01-agentic-rag/lessons/01-intro.md


In [28]:
df_ground_truth.shape

(360, 2)

In [29]:
ground_truth = df_ground_truth.to_dict(orient="records")

ground_truth[0]

{'question': "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?",
 'filename': '01-agentic-rag/lessons/01-intro.md'}

In [30]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

295

In [32]:
from embedder import Embedder

embedder = Embedder()

In [33]:
chunk_texts = [chunk["content"] for chunk in chunks]

X = embedder.encode_batch(chunk_texts)

type(X), X.shape

(numpy.ndarray, (295, 384))

In [34]:
from minsearch import VectorSearch

vector_index = VectorSearch(
    keyword_fields=["filename", "start"]
)

vector_index.fit(X, chunks)

### **Q2**

In [38]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename", "start"]
)

text_index.fit(chunks)

In [39]:
def text_search(query, num_results=5):
    return text_index.search(
        query=query,
        num_results=num_results
    )


def vector_search(query, num_results=5):
    v_query = embedder.encode(query)

    return vector_index.search(
        query_vector=v_query,
        num_results=num_results
    )

In [40]:
q = ground_truth[0]["question"]
q

"What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"

In [41]:
text_results_q2 = text_search(q)

[(r["filename"], r["start"]) for r in text_results_q2]

[('01-agentic-rag/lessons/03-rag.md', 3000),
 ('01-agentic-rag/lessons/13-function-calling.md', 1000),
 ('01-agentic-rag/lessons/03-rag.md', 2000),
 ('01-agentic-rag/lessons/13-function-calling.md', 2000),
 ('01-agentic-rag/lessons/01-intro.md', 0)]

In [42]:
vector_results_q3 = vector_search(q)

[(r["filename"], r["start"]) for r in vector_results_q3]

[('01-agentic-rag/lessons/01-intro.md', 0),
 ('04-evaluation/lessons/11-evaluation-intro.md', 0),
 ('04-evaluation/lessons/12-rag-answers.md', 2000),
 ('01-agentic-rag/lessons/10-rag-next-steps.md', 0),
 ('06-best-practices/lessons/01-intro.md', 0)]

In [43]:
def evaluate(ground_truth, search_function):
    results = []

    for doc in ground_truth:
        question = doc["question"]
        filename = doc["filename"]

        search_results = search_function(question)

        results.append({
            "question": question,
            "filename": filename,
            "results": search_results
        })

    return results

In [44]:
text_search_results = evaluate(
    ground_truth,
    text_search
)

In [45]:
len(text_search_results)

360

In [46]:
def compute_relevance(search_result):
    expected_filename = search_result["filename"]
    results = search_result["results"]

    return [
        int(result["filename"] == expected_filename)
        for result in results
    ]


def hit_rate(search_results):
    cnt = 0

    for search_result in search_results:
        relevance = compute_relevance(search_result)

        if sum(relevance) > 0:
            cnt += 1

    return cnt / len(search_results)


def mrr(search_results):
    total_score = 0.0

    for search_result in search_results:
        relevance = compute_relevance(search_result)

        for rank, rel in enumerate(relevance):
            if rel == 1:
                total_score += 1 / (rank + 1)
                break

    return total_score / len(search_results)

### **Q4**

In [47]:
text_hit_rate = hit_rate(text_search_results)
text_mrr = mrr(text_search_results)

text_hit_rate, text_mrr

(0.7583333333333333, 0.5942592592592594)

In [48]:
vector_search_results = evaluate(
    ground_truth,
    vector_search
)

### **Q5**

In [49]:
vector_hit_rate = hit_rate(vector_search_results)
vector_mrr = mrr(vector_search_results)

vector_hit_rate, vector_mrr

(0.725, 0.5486111111111112)

In [50]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [53]:
def hybrid_search(query, num_results=5, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)

    return rrf(
        [text_results, vector_results],
        k=k,
        num_results=num_results
    )

In [54]:
hybrid_search_results = evaluate(
    ground_truth,
    hybrid_search
)

In [55]:
hybrid_hit_rate = hit_rate(hybrid_search_results)
hybrid_mrr = mrr(hybrid_search_results)

hybrid_hit_rate, hybrid_mrr

(0.8361111111111111, 0.637916666666667)

### **Q6**

In [56]:
def make_hybrid_search(k):
    def search(query):
        return hybrid_search(query, k=k)
    return search


for k in [1, 50, 100, 200]:
    results = evaluate(
        ground_truth,
        make_hybrid_search(k)
    )

    print(k, hit_rate(results), mrr(results))

1 0.8388888888888889 0.6481944444444449
50 0.8361111111111111 0.637916666666667
100 0.8361111111111111 0.637916666666667
200 0.8361111111111111 0.637916666666667
